In [15]:
import numpy as np
from scipy.special import logsumexp

from aeons.utils import generate_Xs, logXf_formula

def sigma_squared_analytic(d, X_i, logL_i):
    """Sigma squared as a function of d and the live points at a certain iteration i"""
    n = len(X_i)
    logsum = np.sum(logL_i)
    sum_X_4d = np.sum(X_i**(4/d))
    sum_X_2d = np.sum(X_i**(2/d))
    sum_log_X_2d = np.sum(X_i**(2/d) * logL_i)
    numerator = n * sum_X_4d - sum_X_2d**2
    denominator = 2 * logsum * sum_X_2d - 2*n*sum_log_X_2d
    return numerator/denominator


def logLmax_analytic(d, X_i, logL_i):
    """Returns logLmax as a function of d and the live points at a certain iteration i"""
    n = len(X_i)
    logsum = np.sum(logL_i)
    sum_X_2d = np.sum(X_i**(2/d))
    return 1/n * logsum + 1/(2*n*sigma_squared_analytic(d, X_i, logL_i)) * sum_X_2d


def params_from_d(logLdata, Xdata, d):
    """Calculates (logLmax, sigma) from d using analytic expressions"""
    sigma = np.sqrt(sigma_squared_analytic(d, Xdata, logLdata))
    logL_max = logLmax_analytic(d, Xdata, logLdata)
    return [logL_max, d, sigma]

Get minimal code required for a working function that calculates the endpoint

In [27]:
from anesthetic import NestedSamples

def get_logbeta_post(points, ndead):
    betas = np.logspace(-5, 1, 1000)
    logX = points.logX()
    logL = points.logL
    logXs = logX.iloc[ndead]
    logLs = logL.iloc[ndead]
    logLbetasX = betas * logLs + logXs - points.logZ(beta=betas) + np.log(betas)
    logprob = logLbetasX - logsumexp(logLbetasX)
    mean = np.sum(np.exp(logprob)*np.log(betas))
    var = np.sum(np.exp(logprob)*(np.log(betas)-mean)**2)
    return mean, np.sqrt(var)

def get_d_G_post(points, ndead, Nset=25):
    logbeta_mean, logbeta_std = get_logbeta_post(points, ndead)
    betas_post = np.exp(np.random.normal(logbeta_mean, logbeta_std, Nset))
    d_G_post = points.d_G(beta=betas_post)
    return d_G_post.values

def logXfs(ndead, logL, logL_birth, Nset=25):
    points = NestedSamples(logL=logL, logL_birth=logL_birth)
    nk = points.nlive
    X_mean = np.cumprod(nk/(nk+1))
    logZ_dead = points[:ndead].logZ()
    d_G = get_d_G_post(points, ndead)
    mean, std = d_G.mean(), d_G.std() # Summarise d_G with its mean and std
    
    logXf_set = np.zeros(Nset)
    for i in range(Nset):
        # Take a sample of X
        X = generate_Xs(nk)
        
        # Repeat until a valid logXf is found
        logXf_i = np.nan
        while np.isnan(logXf_i):
            # Sample from the posterior of d_G
            d = np.random.normal(mean, std)
            # Analytically compute the parameters of the regression to the logL-X curve
            theta = params_from_d(logL[ndead:], X[ndead:], d)
            # Compute the logXf for this d
            logXf_i = logXf_formula(theta, logZ_dead, X_mean[ndead])
        logXf_set[i] = logXf_i
    
    return logXf_set

In [23]:
from aeons.utils import get_samples

name, samples = get_samples('gauss_8')

In [26]:
logL, logL_birth = samples.logL, samples.logL_birth
NestedSamples(logL=logL, logL_birth=logL_birth)

,,logL,logL_birth,nlive
,weights,,,
0,0.000000e+00,-4999.776514,-inf,500
1,0.000000e+00,-4995.072848,-inf,500
2,0.000000e+00,-4992.686941,-inf,500
3,0.000000e+00,-4992.573099,-inf,500
4,0.000000e+00,-4990.915448,-inf,500
...,...,...,...,...
26693,2.084661e-09,-0.002773,-0.010883,5
26694,2.085338e-09,-0.002448,-0.011262,4
26695,2.086643e-09,-0.001822,-0.010834,3


In [41]:
import numpy as np
from numpy import pi, log, sqrt
import pypolychord
from pypolychord.settings import PolyChordSettings
from pypolychord.priors import UniformPrior

nDims = 3
nDerived = 1
nlive = 100

functionName = 'gaussian'

# param : array
def likelihood(theta):
    """ Simple Gaussian Likelihood"""

    sigma = 0.1
    nDims = len(theta)

    r2 = sum(theta**2)

    logL = -log(2*pi*sigma*sigma)*nDims/2.0
    logL += -r2/2/sigma/sigma

    return logL, [r2] # float, array-like

# param : array
def prior(hypercube):
    """ Uniform prior from [-1,1]^D. """
    return UniformPrior(-1, 1)(hypercube) # array

# param : array, array, array, float, float
def dumper(live, dead, logweights, logZ, logZerr):
    if len(live) == 0:
        return
    points = np.concatenate([live, dead])
    logXf_set = logXfs(len(dead), points[:,-1], points[:,-2], Nset=25)
    print(logXf_set)
    iterations_set = logXf_set * -nlive
    iteration_f, iteration_f_std = iterations_set.mean() + nlive, iterations_set.std()
    print(f"Estimated iterations to finish: {iteration_f:.0f} +/- {iteration_f_std:.0f}")
    # Make progress bar based on iteration_f and current iteration, len(dead), with uncertainty
    print(f"Progress: {len(dead)/iteration_f * 100:.0f} +/- {(len(dead)/iteration_f - len(dead)/(iteration_f + iteration_f_std)) * 100:.0f}")
    print(f"True progress: {len(dead)/1425 * 100:.0f}")

settings = PolyChordSettings(nDims, nDerived) #settings is an object
settings.file_root = functionName #string
settings.do_clustering = False 
settings.read_resume = False
settings.nlive = nlive

output = pypolychord.run_polychord(likelihood, nDims, nDerived, settings, prior, dumper)



PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.20.2
  release: 1st June 2021
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     100
nDims    :       3
nDerived :       1
Synchronous parallelisation
Generating equally weighted posteriors
Generating weighted posteriors
Writing a resume file to chains/gaussian.resume

generating live points


all live points generated

Speed  1 =  0.600E-04 seconds
number of repeats:           15
started sampling



/tmp/ipykernel_21714/932790974.py:28: RuntimeWarning: invalid value encountered in sqrt
  sigma = np.sqrt(sigma_squared_analytic(d, Xdata, logLdata))
/home/zixiao/Documents/III/project/aeons/aeons/utils.py:168: RuntimeWarning: invalid value encountered in log
  logdead = logZdead - logLmax - (d/2)*np.log(2) - d*np.log(sigma) + np.log(2/d)


[-11.65737447  -4.59037715 -13.53531435  -5.18718263 -12.24914169
  -7.81453496 -14.87880795 -13.45857951 -16.22549259 -16.06521301
 -11.2764644   -8.52372377  -5.6366357   -5.71127634 -13.9659613
  -7.90564442 -11.66496686 -11.50036544  -5.60782611 -13.29449271
 -15.97018044 -15.11464135  -7.47702892 -12.54611213  -7.47305851]
Estimated iterations to finish: 1177 +/- 375
Progress: 9 +/- 2
True progress: 7
________________
lives      |100 |
phantoms   |932 |
posteriors |101 |
equals     |  9 |
‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
ncluster   =       1 /       1
ndead      =                 101
nposterior =                 101
nequals    =                  11
nlike      =                2593
<nlike>    =          24.93   (           1.66 per slice )
log(Z)     =          -39.96 +/-  0.27
log(Z_1)   =          -39.96 +/-  0.27 (still evaluating)



[-12.70164659  -9.87897266 -10.46590828 -12.24163479 -13.1679562
 -13.87856276 -14.24468273 -11.50375115 -11.8627827  -14.72540141
 -13.24241855  -9.38394381  -8.